# Lab 07
MSSV: 22302421

Name: Nguyễn Phúc Minh Châu

In [6]:
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    'order_id': [1,2,3,4,5,6],
    'customer_id': ['C1','C2','C1','C3','C2','C4'],
    'product': ['Laptop','Phone','Tablet','Laptop','Tablet','Phone'],
    'price': [1000,500,300,1200,350,450],
    'quantity': [1,2,1,1,3,2],
    'date': pd.to_datetime(['2024-01-01','2024-01-02','2024-01-05','2024-02-01','2024-02-10','2024-03-01'])
})


customers = pd.DataFrame({
    'customer_id': ['C1','C2','C3','C4'],
    'city': ['HCM','HN','HCM','DN'],
    'segment': ['VIP','Normal','VIP','Normal']
})

orders

,order_id,customer_id,product,price,quantity,date
0,1,C1,Laptop,1000,1,2024-01-01
1,2,C2,Phone,500,2,2024-01-02
2,3,C1,Tablet,300,1,2024-01-05
3,4,C3,Laptop,1200,1,2024-02-01
4,5,C2,Tablet,350,3,2024-02-10
5,6,C4,Phone,450,2,2024-03-01


In [7]:
customers

,customer_id,city,segment
0,C1,HCM,VIP
1,C2,HN,Normal
2,C3,HCM,VIP
3,C4,DN,Normal


### Exercise 1: Calculate the total revenue per customer (merge + groupby).

In [8]:
orders['total_revenue'] = orders['price'] * orders['quantity']
customer_revenue = orders.groupby('customer_id')['total_revenue'].sum().reset_index()
customer_revenue

,customer_id,total_revenue
0,C1,1300
1,C2,2050
2,C3,1200
3,C4,900


### Exercise 2: Select the top 2 customers with the highest revenue.

In [9]:
top_customers = customer_revenue.sort_values(by='total_revenue', ascending=False).head(2)
top_customers

,customer_id,total_revenue
1,C2,2050
0,C1,1300


### Exercise 3: Calculate total revenue by city (multi-group).

In [10]:
# Exercise 3: Calculate total revenue by city (multi-group).
orders_with_city = orders.merge(customers[['customer_id', 'city']], on='customer_id', how='left')
revenue_by_city = orders_with_city.groupby('city')['total_revenue'].sum().reset_index()
revenue_by_city

,city,total_revenue
0,DN,900
1,HCM,2500
2,HN,2050


### Exercise 4: Pivot table for monthly revenue 
Create table: row = month, column = product, value = revenue.

In [11]:
orders['month'] = orders['date'].dt.to_period('M')
pivot_table = orders.pivot_table(index='month', columns='product', values='total_revenue', aggfunc='sum', fill_value=0)
pivot_table

product,Laptop,Phone,Tablet
month,,,
2024-01,1000,1000,300
2024-02,1200,0,1050
2024-03,0,900,0


### Exercise 5: Find customers who have placed more than one order.

In [12]:
order_counts = orders.groupby('customer_id')['order_id'].count().reset_index()
repeat_customers = order_counts[order_counts['order_id'] > 1]
repeat_customers

,customer_id,order_id
0,C1,2
1,C2,2


### Exercise 6: Divide customers into 3 groups (binning): Low, Medium, High.

In [13]:
customer_revenue['revenue_bin'] = pd.qcut(customer_revenue['total_revenue'], q=3, labels=['Low', 'Medium', 'High'])
customer_revenue

,customer_id,total_revenue,revenue_bin
0,C1,1300,Medium
1,C2,2050,High
2,C3,1200,Low
3,C4,900,Low


### Exercise 7: If there is a missing value in the price, replace it with the average.
orders.loc[2, 'price'] = np.nan

In [14]:
orders.loc[2, 'price'] = np.nan
average_price = orders['price'].mean()
orders['price'].fillna(average_price, inplace=True)
orders

/tmp/ipykernel_102073/3144291870.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  orders['price'].fillna(average_price, inplace=True)


,order_id,customer_id,product,price,quantity,date,total_revenue,month
0,1,C1,Laptop,1000.0,1,2024-01-01,1000,2024-01
1,2,C2,Phone,500.0,2,2024-01-02,1000,2024-01
2,3,C1,Tablet,NaN,1,2024-01-05,300,2024-01
3,4,C3,Laptop,1200.0,1,2024-02-01,1200,2024-02
4,5,C2,Tablet,350.0,3,2024-02-10,1050,2024-02
5,6,C4,Phone,450.0,2,2024-03-01,900,2024-03


### Exercise 8: Look for orders with unusually high prices (Detecting outliers (simple z-score)).

In [15]:
z_scores = (orders['price'] - orders['price'].mean()) / orders['price'].std()
outliers = orders[np.abs(z_scores) > 3]
print(outliers)

Empty DataFrame
Columns: [order_id, customer_id, product, price, quantity, date, total_revenue, month]
Index: []


### Exercise 9: Three orders are randomly selected.

In [16]:
sampled_orders = orders.sample(n=3, random_state=42)
sampled_orders

,order_id,customer_id,product,price,quantity,date,total_revenue,month
0,1,C1,Laptop,1000.0,1,2024-01-01,1000,2024-01
1,2,C2,Phone,500.0,2,2024-01-02,1000,2024-01
5,6,C4,Phone,450.0,2,2024-03-01,900,2024-03


### Exercise 10: Calculate total revenue by month and sort by time (Analyze time series (monthly trends)).

In [17]:
monthly_revenue = orders.groupby('month')['total_revenue'].sum().reset_index()
monthly_revenue.sort_values(by='month', inplace=True)
monthly_revenue

,month,total_revenue
0,2024-01,2300
1,2024-02,2250
2,2024-03,900
